# Hurricane Helene (2024) — AI Model Track Comparison

Compare tropical cyclone tracks from **GraphCast**, **FCN3**, and **AIFS**
against ERA5 reanalysis for Hurricane Helene, at three lead times before
landfall (T−72 h, T−168 h, T−240 h).

**Verification target:** Helene's Florida Big Bend landfall,
snapped to `2024-09-27 00:00 UTC`.

| Lead | Init time (UTC)  | Steps (6 h) |
|------|------------------|-------------|
| 72 h | 2024-09-24 00:00 | 12          |
| 168 h| 2024-09-20 00:00 | 28          |
| 240 h| 2024-09-17 00:00 | 40          |

In [10]:
import os

# Must be set BEFORE importing torch.
os.environ["FORCE_CUDA_EXTENSION"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.9"
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["EARTH2STUDIO_CACHE"] = os.path.expanduser("~/.cache/earth2studio")
os.environ["CUFILE_ENV_PATH_JSON"] = "/dev/null"  # Disable GPU Direct Storage (not available in containers)

for k in ["FORCE_CUDA_EXTENSION", "TORCH_CUDA_ARCH_LIST", "CUDA_HOME", "EARTH2STUDIO_CACHE"]:
    print(f"  {k} = {os.environ[k]}")
print("DONE")

  FORCE_CUDA_EXTENSION = 1
  TORCH_CUDA_ARCH_LIST = 8.9
  CUDA_HOME = /usr/local/cuda
  EARTH2STUDIO_CACHE = /home/jovyan/.cache/earth2studio
DONE

In [11]:
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import torch
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tqdm import tqdm

from earth2studio.models.px import FCN3, AIFS
from earth2studio.models.dx import TCTrackerWuDuan
from earth2studio.data import WB2ERA5, ARCO, fetch_data, prep_data_array
from earth2studio.io import ZarrBackend
from earth2studio.utils.time import to_time_array
from earth2studio.utils.coords import map_coords

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Done")

Device: cuda
GPU   : NVIDIA L40
VRAM  : 47.7 GB
Done

In [12]:
# ── Storm & experiment configuration ──────────────────────────────────
STORM_NAME = "Helene"

# Landfall snapped to nearest 6h boundary.
VERIFICATION_TIME = datetime(2024, 9, 27, 0)

# Lead times in hours before VERIFICATION_TIME.
LEAD_TIMES_H = [72, 168, 240]

# Derived: init time and step count for each lead.
INIT_TIMES = {h: VERIFICATION_TIME - timedelta(hours=h) for h in LEAD_TIMES_H}
N_STEPS    = {h: h // 6 for h in LEAD_TIMES_H}

# Output directory — all .pt track files and plots go here.
OUT_DIR = Path("./outputs/helene")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Plot extent (lon_min, lon_max, lat_min, lat_max) for Helene's domain.
PLOT_EXTENT = (-100, -65, 15, 40)

# Landfall point for the star marker.
LANDFALL_LAT = 30.0
LANDFALL_LON = -83.7

# Consistent model colors across all notebooks in this study.
# GraphCast color reserved for when its separate JAX env is ready.
MODEL_COLORS = {
    "FCN3":      "#ff7f0e",  # orange
    "AIFS":      "#2ca02c",  # green
    "GraphCast": "#1f77b4",  # blue (from separate JAX environment)
    "ERA5":      "#000000",  # black (reanalysis truth)
}

# Print the resolved init times.
for h in LEAD_TIMES_H:
    print(f"  T-{h:>3}h : init={INIT_TIMES[h]}  steps={N_STEPS[h]}")

  T- 72h : init=2024-09-24 00:00:00  steps=12
  T-168h : init=2024-09-20 00:00:00  steps=28
  T-240h : init=2024-09-17 00:00:00  steps=40

## How the tracking works

Unlike post-hoc tracking on saved forecasts, `TCTrackerWuDuan` runs
**during** inference. At each 6 h autoregressive step:

1. The prognostic model produces the next atmospheric state.
2. That state is passed to the tracker, which updates its internal
   path buffer with detected cyclone positions.
3. After the final step, the tracker's output tensor contains all
   detected tracks: shape `(batch, num_paths, time_steps, 2)` where
   the last dimension is `[lat, lon]`.

This is more efficient than saving full global fields to disk and
tracking afterward — and it's the pattern used in the Earth2Studio
cyclone tracking example.

In [13]:
def run_tc_inference(prognostic, data_source, start_time, nsteps, save_path):
    """Run a prognostic model through TCTrackerWuDuan and save tracks.

    Args:
        prognostic:   Loaded Earth2Studio prognostic model.
        data_source:  Data source for initial conditions (e.g. WB2ERA5).
        start_time:   datetime — forecast initialization time.
        nsteps:       Number of 6h autoregressive steps.
        save_path:    Path to save the output .pt track tensor.

    Returns:
        torch.Tensor of shape (batch, num_paths, time_steps, 2).
    """
    # Skip if already cached — avoids re-running expensive inference.
    if Path(save_path).exists():
        print(f"  cached → {save_path}")
        return torch.load(str(save_path), map_location="cpu")

    # Move model to GPU.
    prognostic = prognostic.to(DEVICE)

    # Fresh tracker instance per run — reset_path_buffer clears any
    # leftover state from a previous call.
    tracker = TCTrackerWuDuan()
    tracker.reset_path_buffer()

    # Fetch initial condition from the data source.
    # fetch_data returns (tensor, coords_dict) on the target device.
    x, coords = fetch_data(
        source=data_source,
        time=to_time_array([start_time]),
        variable=prognostic.input_coords()["variable"],
        lead_time=prognostic.input_coords()["lead_time"],
        device=DEVICE,
    )
    # Align coords to the model's expected layout.
    x, coords = map_coords(x, coords, prognostic.input_coords())

    # Step through the autoregressive iterator.
    # Each iteration: model produces next state → tracker ingests it.
    iterator = prognostic.create_iterator(x, coords)
    for step, (x, coords) in tqdm(enumerate(iterator), total=nsteps + 1):
        # map_coords aligns the model output to tracker's expected vars.
        x, coords = map_coords(x, coords, tracker.input_coords())
        # Tracker call: updates internal path buffer, returns current tracks.
        output, _ = tracker(x, coords)
        output = output[:, 0]  # take the first (and usually only) batch dim
        if step == nsteps:
            break

    # Save tracks as a .pt tensor for fast reload.
    tracks = output.cpu()
    torch.save(tracks, str(save_path))
    print(f"  saved  → {save_path}")
    return tracks

In [14]:
def run_era5_truth(data_source, start_time, nsteps, save_path):
    """Track ERA5 reanalysis through the same TCTrackerWuDuan.

    Unlike forecast models, ERA5 doesn't need autoregressive stepping.
    We just load each 6h time slice and feed it to the tracker sequentially.
    This ensures truth tracks use the identical tracking algorithm.

    Args:
        data_source:  ERA5 data source (WB2ERA5 or ARCO).
        start_time:   datetime — beginning of the truth window.
        nsteps:       Number of 6h steps to cover.
        save_path:    Path to save the output .pt track tensor.

    Returns:
        torch.Tensor of shape (batch, num_paths, time_steps, 2).
    """
    if Path(save_path).exists():
        print(f"  cached → {save_path}")
        return torch.load(str(save_path), map_location="cpu")

    tracker = TCTrackerWuDuan()
    tracker.reset_path_buffer()

    # Build the list of valid times: every 6h from start through start + nsteps*6h.
    times = [start_time + timedelta(hours=6 * i) for i in range(nsteps + 1)]

    for step, time in tqdm(enumerate(times), total=len(times)):
        # prep_data_array converts a data source call into (tensor, coords).
        da = data_source(time, tracker.input_coords()["variable"])
        x, coords = prep_data_array(da, device=DEVICE)
        output, _ = tracker(x, coords)

    tracks = output.cpu()
    torch.save(tracks, str(save_path))
    print(f"  saved  → {save_path}")
    return tracks

In [15]:
# ARCO = Analysis-Ready, Cloud Optimized ERA5 on GCS.
# Used for both model initialization and ERA5 truth tracking.
# WB2ERA5 has a hard cutoff of 2023-01-11 — all Helene init times are in Sep 2024.
data_source = ARCO(cache=True)

2026-05-01 18:54:37.975 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/.zattrs to local cache2026-05-01 18:54:38.554 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/.zgroup to local cache
2026-05-01 18:54:38.688 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/.zmetadata to local cache2026-05-01 18:54:39.008 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/level/0 to local cache2026-05-01 18:54:39.308 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/model-level-1h-0p25deg.zarr-v1/.zattrs to local cache
2026-05-01 18:54:39.500 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Co

## FCN3 forecasts

FCN3 uses BF16 automatic mixed precision — actual VRAM usage is ~38–42 GB
on the L40 despite the 80 GB listed in the docs (which reflects FP32 peak).
Ensure torch-harmonics compiled with CUDA extensions (verified at startup).

In [16]:
%%time
fcn3 = FCN3.load_model(FCN3.load_default_package())

CPU times: user 10.5 s, sys: 5.58 s, total: 16.1 s
Wall time: 10.9 s

In [17]:
%%time
fcn3_tracks_72 = run_tc_inference(
    fcn3, data_source,
    INIT_TIMES[72], N_STEPS[72],
    OUT_DIR / "fcn3_72h.pt",
)

Fetching ARCO data:   0%|          | 0/72 [00:00<?, ?it/s]

2026-05-01 18:55:41.689 | DEBUG    | earth2studio.data.arco:fetch_array:297 - Fetching ARCO zarr array for variable: v300 at 2024-09-24T00:00:00
2026-05-01 18:55:41.710 | DEBUG    | earth2studio.data.arco:fetch_array:297 - Fetching ARCO zarr array for variable: q500 at 2024-09-24T00:00:00
2026-05-01 18:55:41.712 | DEBUG    | earth2studio.data.arco:fetch_array:297 - Fetching ARCO zarr array for variable: t250 at 2024-09-24T00:00:00
2026-05-01 18:55:41.713 | DEBUG    | earth2studio.data.arco:fetch_array:297 - Fetching ARCO zarr array for variable: t500 at 2024-09-24T00:00:00
2026-05-01 18:55:41.714 | DEBUG    | earth2studio.data.arco:fetch_array:297 - Fetching ARCO zarr array for variable: z925 at 2024-09-24T00:00:00
2026-05-01 18:55:41.715 | DEBUG    | earth2studio.data.arco:fetch_array:297 - Fetching ARCO zarr array for variable: u700 at 2024-09-24T00:00:00
2026-05-01 18:55:41.716 | DEBUG    | earth2studio.data.arco:fetch_array:297 - Fetching ARCO zarr array for variable: z100 at 2024-

Fetching ARCO data:   0%|          | 0/72 [00:00<?, ?it/s]

2026-05-01 18:55:42.269 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/v_component_of_wind/1093368.0.0.0 to local cache
2026-05-01 18:55:42.272 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/temperature/1093368.0.0.0 to local cache
2026-05-01 18:55:42.274 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/temperature/1093368.0.0.0 to local cache
2026-05-01 18:55:42.276 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/temperature/1093368.0.0.0 to local cache
2026-05-01 18:55:42.278 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/u_component_of_wind/1093368.

Fetching ARCO data:   0%|          | 0/72 [00:00<?, ?it/s]

2026-05-01 18:55:42.486 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/geopotential/1093368.0.0.0 to local cache
2026-05-01 18:55:42.506 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/v_component_of_wind/1093368.0.0.0 to local cache
2026-05-01 18:55:42.515 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/u_component_of_wind/1093368.0.0.0 to local cache
2026-05-01 18:55:42.517 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/specific_humidity/1093368.0.0.0 to local cache
2026-05-01 18:55:42.518 | DEBUG    | earth2studio.data.utils:_make_local_details:770 - Copying /gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/temperature/1

  8%|▊         | 1/13 [01:30<18:01, 90.11s/it]

CPU times: user 40 s, sys: 20.4 s, total: 1min
Wall time: 2min 11s

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.37 GiB. GPU 0 has a total capacity of 44.39 GiB of which 11.22 GiB is free. Including non-PyTorch memory, this process has 33.16 GiB memory in use. Of the allocated memory 32.29 GiB is allocated by PyTorch, and 350.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
%%time
fcn3_tracks_168 = run_tc_inference(
    fcn3, data_source,
    INIT_TIMES[168], N_STEPS[168],
    OUT_DIR / "fcn3_168h.pt",
)

In [ ]:
%%time
fcn3_tracks_240 = run_tc_inference(
    fcn3, data_source,
    INIT_TIMES[240], N_STEPS[240],
    OUT_DIR / "fcn3_240h.pt",
)

## AIFS forecasts

AIFS (ECMWF) — GNN encoder + sliding-window transformer processor.
First ML model fully operational at a major NWP center (Feb 2025).

In [ ]:
%%time
aifs = AIFS.load_model(AIFS.load_default_package())

In [ ]:
%%time
aifs_tracks_72 = run_tc_inference(
    aifs, data_source,
    INIT_TIMES[72], N_STEPS[72],
    OUT_DIR / "aifs_72h.pt",
)

In [ ]:
%%time
aifs_tracks_168 = run_tc_inference(
    aifs, data_source,
    INIT_TIMES[168], N_STEPS[168],
    OUT_DIR / "aifs_168h.pt",
)

In [ ]:
%%time
aifs_tracks_240 = run_tc_inference(
    aifs, data_source,
    INIT_TIMES[240], N_STEPS[240],
    OUT_DIR / "aifs_240h.pt",
)

## ERA5 truth tracks

Run the same `TCTrackerWuDuan` on ERA5 reanalysis for each lead-time
window. This ensures truth and forecasts use an identical tracking
algorithm — no confound from different tracker implementations.

In [ ]:
%%time
# One truth track per lead time. Each covers [init_time, verification_time].
era5_tracks_72 = run_era5_truth(
    data_source, INIT_TIMES[72], N_STEPS[72],
    OUT_DIR / "era5_72h.pt",
)

In [ ]:
%%time
era5_tracks_168 = run_era5_truth(
    data_source, INIT_TIMES[168], N_STEPS[168],
    OUT_DIR / "era5_168h.pt",
)

In [ ]:
%%time
era5_tracks_240 = run_era5_truth(
    data_source, INIT_TIMES[240], N_STEPS[240],
    OUT_DIR / "era5_240h.pt",
)

## Load saved tracks
Reload-safe: works whether you just ran inference or restarted the kernel.

In [ ]:
# Reload all .pt files — safe to re-run after a kernel restart.
# GraphCast tracks (from separate JAX env) will be appended here once available.
fcn3_tracks_72  = torch.load(OUT_DIR / "fcn3_72h.pt",  map_location="cpu")
fcn3_tracks_168 = torch.load(OUT_DIR / "fcn3_168h.pt", map_location="cpu")
fcn3_tracks_240 = torch.load(OUT_DIR / "fcn3_240h.pt", map_location="cpu")

aifs_tracks_72  = torch.load(OUT_DIR / "aifs_72h.pt",  map_location="cpu")
aifs_tracks_168 = torch.load(OUT_DIR / "aifs_168h.pt", map_location="cpu")
aifs_tracks_240 = torch.load(OUT_DIR / "aifs_240h.pt", map_location="cpu")

era5_tracks_72  = torch.load(OUT_DIR / "era5_72h.pt",  map_location="cpu")
era5_tracks_168 = torch.load(OUT_DIR / "era5_168h.pt", map_location="cpu")
era5_tracks_240 = torch.load(OUT_DIR / "era5_240h.pt", map_location="cpu")

print(f"FCN3 T-72h track shape : {fcn3_tracks_72.shape}")
print(f"ERA5 T-72h track shape : {era5_tracks_72.shape}")

## Plot 1 — Track comparison maps

In [ ]:
def plot_tracks(models_list, extent, title, save_path=None):
    """Plot one or more TC track sets on a single Cartopy map.

    Args:
        models_list:  List of dicts, each with keys:
                        name  (str), tracks (Tensor), color (str),
                        lw (float), ls (str — line style).
        extent:       (lon_min, lon_max, lat_min, lat_max).
        title:        Plot title string.
        save_path:    Optional path to save the figure.
    """
    fig, ax = plt.subplots(
        subplot_kw={"projection": ccrs.PlateCarree()},
        figsize=(10, 7),
    )

    # Geographic features.
    ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.4)
    ax.add_feature(cfeature.OCEAN, facecolor="white")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.STATES, linewidth=0.3, linestyle=":")
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    # Plot each model's tracks.
    for m in models_list:
        tracks = m["tracks"]
        # Handle both torch tensors and numpy arrays.
        if hasattr(tracks, "detach"):
            tracks = tracks.detach().cpu().numpy()

        label_set = False
        # tracks shape: (batch, num_paths, time_steps, 2)
        for p in range(tracks.shape[1]):
            lats = tracks[0, p, :, 0]  # lat is index 0
            lons = tracks[0, p, :, 1]  # lon is index 1
            # Filter out NaN positions (no storm detected at that step).
            mask = ~np.isnan(lats) & ~np.isnan(lons)
            if mask.sum() > 2:
                ax.plot(
                    lons[mask], lats[mask],
                    color=m["color"], linewidth=m.get("lw", 2),
                    linestyle=m.get("ls", "-"),
                    label=m["name"] if not label_set else "",
                    transform=ccrs.PlateCarree(), zorder=6,
                )
                label_set = True

    # Landfall marker.
    ax.plot(
        LANDFALL_LON, LANDFALL_LAT,
        marker="*", color="red", markersize=18, markeredgecolor="black",
        linestyle="None", label="Landfall",
        transform=ccrs.PlateCarree(), zorder=10,
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.3)
    gl.top_labels = False
    gl.right_labels = False
    ax.legend(loc="lower right")
    ax.set_title(title, fontsize=13)

    if save_path:
        fig.savefig(str(save_path), dpi=150, bbox_inches="tight")
        print(f"saved → {save_path}")
    plt.show()

In [ ]:
plot_tracks(
    [
        {"name": "ERA5 (truth)", "tracks": era5_tracks_72, "color": MODEL_COLORS["ERA5"], "lw": 2.5, "ls": "-"},
        {"name": "FCN3",         "tracks": fcn3_tracks_72, "color": MODEL_COLORS["FCN3"], "lw": 2.0, "ls": "--"},
        {"name": "AIFS",         "tracks": aifs_tracks_72, "color": MODEL_COLORS["AIFS"], "lw": 2.0, "ls": "--"},
    ],
    extent=PLOT_EXTENT,
    title=f"{STORM_NAME} (2024) — T−72 h track comparison (init {INIT_TIMES[72]:%Y-%m-%d %H}Z)",
    save_path=OUT_DIR / "track_72h.png",
)

In [ ]:
plot_tracks(
    [
        {"name": "ERA5 (truth)", "tracks": era5_tracks_168, "color": MODEL_COLORS["ERA5"], "lw": 2.5, "ls": "-"},
        {"name": "FCN3",         "tracks": fcn3_tracks_168, "color": MODEL_COLORS["FCN3"], "lw": 2.0, "ls": "--"},
        {"name": "AIFS",         "tracks": aifs_tracks_168, "color": MODEL_COLORS["AIFS"], "lw": 2.0, "ls": "--"},
    ],
    extent=PLOT_EXTENT,
    title=f"{STORM_NAME} (2024) — T−168 h track comparison (init {INIT_TIMES[168]:%Y-%m-%d %H}Z)",
    save_path=OUT_DIR / "track_168h.png",
)

In [ ]:
plot_tracks(
    [
        {"name": "ERA5 (truth)", "tracks": era5_tracks_240, "color": MODEL_COLORS["ERA5"], "lw": 2.5, "ls": "-"},
        {"name": "FCN3",         "tracks": fcn3_tracks_240, "color": MODEL_COLORS["FCN3"], "lw": 2.0, "ls": "--"},
        {"name": "AIFS",         "tracks": aifs_tracks_240, "color": MODEL_COLORS["AIFS"], "lw": 2.0, "ls": "--"},
    ],
    extent=PLOT_EXTENT,
    title=f"{STORM_NAME} (2024) — T−240 h track comparison (init {INIT_TIMES[240]:%Y-%m-%d %H}Z)",
    save_path=OUT_DIR / "track_240h.png",
)

## Summary

Runs **FCN3** and **AIFS** TC track forecasts for Hurricane Helene (2024) at T−72h, T−168h, and T−240h, compared against ERA5 reanalysis tracked with the same `TCTrackerWuDuan`.

**GraphCast** is excluded here — JAX ≥ 0.5 is incompatible with the CUDA 13.1 driver on TIDE. It will run in a separate isolated volume and its `.pt` track files will be dropped into `./outputs/helene/` and loaded in the reload cell above.

**Saved outputs in `./outputs/helene/`:**
- 6 forecast track files: `fcn3_72h.pt`, `fcn3_168h.pt`, `fcn3_240h.pt`, `aifs_72h.pt`, `aifs_168h.pt`, `aifs_240h.pt`
- 3 ERA5 truth track files: `era5_72h.pt`, `era5_168h.pt`, `era5_240h.pt`
- 3 track comparison plots: `track_72h.png`, `track_168h.png`, `track_240h.png`

**For Milton and Otis notebooks:** copy this notebook and change only `STORM_NAME`, `VERIFICATION_TIME`, `PLOT_EXTENT`, `LANDFALL_LAT/LON`, and `OUT_DIR`. All helpers and `MODEL_COLORS` carry over unchanged.